# KelpAdam – UTS Pembelajaran Mesin
## Data Preparation & EDA: Preferensi Gaya Belajar (Gasal 2026/2027)

Notebook ini mengolah dataset survei gaya belajar dari dosen sampai siap dianalisis dan siap untuk tahap modelling.

| Bagian | Isi | Status |
|---|---|---|
| 0 | Setup & aturan privasi | ✅ |
| 1 | Data understanding (memahami isi data, mendeteksi masalah) | ✅ |
| 2 | Data cleaning + cleaning log | ⏳ berikutnya |
| 3 | Scoring sesuai teori 4 framework | ⏳ |
| 4 | Data preparation & data dictionary | ⏳ |
| 5 | EDA | ⏳ |
| 6 | Ringkasan, keterbatasan, ekspor output | ⏳ |

**Cara menjalankan:** buka notebook dari folder `notebook/`, lalu pilih *Kernel → Restart & Run All*. Semua path bersifat relatif terhadap folder proyek `UTS/`.

---
# Bagian 0 – Setup

**Aturan yang dipegang di seluruh notebook** (dari spesifikasi proyek):
1. File Excel asli **tidak pernah diubah**. Semua perubahan dilakukan pada salinan di memori lewat kode.
2. Dataset tidak berisi identitas langsung (nama, NPM, email). Kolom `ID` hanya nomor urut.
3. Setiap penghapusan atau perubahan data dicatat di **cleaning log**, beserta jumlah sebelum/sesudah dan alasannya.
4. Outlier **tidak** dihapus otomatis.

In [5]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# --- Path relatif: notebook ada di UTS/notebook/, data di UTS/dataset/ ---
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW_FILE    = PROJECT_DIR / "dataset" / "raw" / "Survei_Gaya_Belajar_IPS_IPK_NO_PII.xlsx"
REPORT_DIR  = PROJECT_DIR / "report"
PREFIX      = "KelpAdam"

print("Folder proyek :", PROJECT_DIR)
print("File data     :", RAW_FILE.name, "| ada?", RAW_FILE.exists())

Folder proyek : d:\Kuliah\Semester 5\Machine Learning\UTS
File data     : Survei_Gaya_Belajar_IPS_IPK_NO_PII.xlsx | ada? True


---
# Bagian 1 – Data Understanding

Tujuan bagian ini: **mengenal data sebelum mengubah apa pun**. Pertanyaan yang dijawab:
- Seberapa besar datanya dan dari mana asalnya?
- Kolom apa saja yang ada, dan masuk kelompok apa?
- Apa tipe datanya, berapa nilai uniknya, dan berapa rentangnya?
- Di mana ada data kosong atau format yang aneh?

## 1.1 Membaca data & bukti asal-usul (provenance)
Hash MD5 adalah "sidik jari" file. Kalau file Excel berubah sedikit saja, hash-nya ikut berubah. Ini bukti bahwa kita bekerja dengan file asli dari dosen.

In [6]:
raw = pd.read_excel(RAW_FILE)
md5 = hashlib.md5(RAW_FILE.read_bytes()).hexdigest()

print(f"Ukuran data : {raw.shape[0]} baris x {raw.shape[1]} kolom")
print(f"Jumlah sheet: {len(pd.ExcelFile(RAW_FILE).sheet_names)}")
print(f"MD5         : {md5}")

Ukuran data : 194 baris x 116 kolom
Jumlah sheet: 1
MD5         : 390ab027d2a950f63a72684d795caa72


## 1.2 Kelompok kolom
Header kolom berupa teks pertanyaan yang panjang, jadi kolom dikelompokkan berdasarkan **posisinya**. Pengelompokan ini disusun dari urutan kolom dan isi pernyataan (lihat dokumen *Aturan Scoring*, Lampiran 2).

In [7]:
cols = list(raw.columns)
assert len(cols) == 116, "Struktur kolom berbeda dari yang diharapkan"

# (nama kelompok, framework, posisi awal, posisi akhir, kode singkat)
GROUPS = [
    ("ID",                  "Identifier",       0,   0,   "ID"),
    ("VARK 16 pertanyaan",  "VARK",             1,   16,  "VARK_Q"),
    ("Activist",            "Honey-Mumford",    17,  24,  "HM_ACT"),
    ("Reflector",           "Honey-Mumford",    25,  32,  "HM_REF"),
    ("Theorist",            "Honey-Mumford",    33,  40,  "HM_THE"),
    ("Pragmatist",          "Honey-Mumford",    41,  48,  "HM_PRA"),
    ("Active",              "Felder-Silverman", 49,  52,  "FS_ACT"),
    ("Reflective",          "Felder-Silverman", 53,  56,  "FS_REF"),
    ("Sensing",             "Felder-Silverman", 57,  60,  "FS_SEN"),
    ("Intuitive",           "Felder-Silverman", 61,  64,  "FS_INT"),
    ("Visual",              "Felder-Silverman", 65,  68,  "FS_VIS"),
    ("Verbal",              "Felder-Silverman", 69,  72,  "FS_VRB"),
    ("Sequential",          "Felder-Silverman", 73,  76,  "FS_SEQ"),
    ("Global",              "Felder-Silverman", 77,  80,  "FS_GLO"),
    ("CE",                  "Kolb",             81,  86,  "KOLB_CE"),
    ("RO",                  "Kolb",             87,  92,  "KOLB_RO"),
    ("AC",                  "Kolb",             93,  98,  "KOLB_AC"),
    ("AE",                  "Kolb",             99,  104, "KOLB_AE"),
    ("IPS per semester",    "Nilai akademik",   105, 114, "IPS"),
    ("IPK",                 "Nilai akademik",   115, 115, "IPK"),
]

ringkas = pd.DataFrame(
    [(fw, nama, f"{a}–{b}", b - a + 1, cols[a][:70]) for nama, fw, a, b, _ in GROUPS],
    columns=["Framework", "Kelompok", "Posisi kolom", "Jumlah kolom", "Contoh header (dipotong)"],
)
display(ringkas)

VARK_COLS   = cols[1:17]
LIKERT_COLS = cols[17:105]
IPS_COLS    = cols[105:115]
IPK_COL     = cols[115]
SURVEY_COLS = VARK_COLS + LIKERT_COLS

# Kode pendek untuk setiap kolom (dipakai untuk tampilan, lalu untuk rename di C1)
nama_baru = {}
for nama, fw, a, b, kode in GROUPS:
    for k, pos in enumerate(range(a, b + 1), 1):
        asli = cols[pos]
        if kode in ("ID", "IPK"):
            baru = kode
        elif kode == "IPS":
            tahun, smt = asli.split()[1], asli.split()[-1]      # "IPS 2025 Semester 2" -> IPS_2025_S2
            baru = f"IPS_{tahun}_S{smt}"
        elif kode == "VARK_Q":
            baru = f"VARK_Q{k:02d}"
        else:
            baru = f"{kode}_{k:02d}"
        nama_baru[asli] = baru

,Framework,Kelompok,Posisi kolom,Jumlah kolom,Contoh header (dipotong)
0,Identifier,ID,0–0,1,ID
1,VARK,VARK 16 pertanyaan,1–16,16,"Ketika mempelajari topik baru yang cukup kompleks, saya paling terbant"
2,Honey-Mumford,Activist,17–24,8,Saya menikmati aktivitas belajar baru meskipun belum sepenuhnya tahu b
3,Honey-Mumford,Reflector,25–32,8,Saya suka mengamati terlebih dahulu sebelum mengambil kesimpulan.
4,Honey-Mumford,Theorist,33–40,8,"Saya merasa nyaman jika suatu materi memiliki konsep, model, atau kera"
5,Honey-Mumford,Pragmatist,41–48,8,Saya tertarik pada ide yang dapat segera diterapkan untuk memecahkan m
6,Felder-Silverman,Active,49–52,4,Saya memahami materi lebih cepat ketika dapat mendiskusikannya atau me
7,Felder-Silverman,Reflective,53–56,4,Saya lebih suka memikirkan materi secara mendalam sebelum membicarakan
8,Felder-Silverman,Sensing,57–60,4,"Saya lebih menyukai fakta, data, contoh nyata, dan prosedur yang jelas"
9,Felder-Silverman,Intuitive,61–64,4,"Saya tertarik pada kemungkinan, pola, konsep baru, atau pendekatan yan"


## 1.3 Daftar lengkap nama kolom & tipe data
Seluruh 116 kolom beserta kode pendeknya, tipe data, jumlah sel terisi/kosong, dan jumlah nilai unik.

In [8]:
daftar_kolom = pd.DataFrame({
    "Posisi": range(len(cols)),
    "Kode pendek": [nama_baru[c] for c in cols],
    "Nama kolom asli": cols,
    "Tipe data": [str(raw[c].dtype) for c in cols],
    "Terisi": raw.notna().sum().values,
    "Kosong": raw.isna().sum().values,
    "Nilai unik": raw.nunique().values,
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", 110):
    display(daftar_kolom)

,Posisi,Kode pendek,Nama kolom asli,Tipe data,Terisi,Kosong,Nilai unik
0,0,ID,ID,int64,194,0,194
1,1,VARK_Q01,"Ketika mempelajari topik baru yang cukup kompleks, saya paling terbantu jika...",str,192,2,41
2,2,VARK_Q02,"Saat harus memahami prosedur penggunaan suatu perangkat atau aplikasi, saya cenderung...",str,192,2,35
3,3,VARK_Q03,"Untuk mengingat materi menjelang ujian, saya biasanya lebih suka...",str,192,2,30
4,4,VARK_Q04,"Ketika dosen menjelaskan konsep abstrak, saya lebih mudah memahami bila...",str,192,2,33
5,5,VARK_Q05,"Saat menerima tugas baru, saya biasanya ingin terlebih dahulu...",str,192,2,35
6,6,VARK_Q06,"Jika saya tersesat di tempat yang belum dikenal, saya lebih memilih...",str,192,2,39
7,7,VARK_Q07,"Saat mempelajari data atau hasil penelitian, saya paling terbantu oleh...",str,192,2,39
8,8,VARK_Q08,"Jika harus menjelaskan ide kepada teman, saya cenderung...",str,192,2,32
9,9,VARK_Q09,"Dalam kelas daring, saya paling tertarik ketika materi menyediakan...",str,192,2,31


**Cara membaca tipe data:**
- `int64` = bilangan bulat (ID).
- `str`/`object` = teks (jawaban VARK).
- `float64` = angka desimal. Pernyataan skala 1–5 tersimpan sebagai desimal karena ada sel kosong (pandas mengubah kolom berisi `NaN` menjadi float), padahal isinya bilangan bulat. Tipe ini diperbaiki di cleaning (C3).

## 1.4 Contoh baris
Lima baris pertama untuk beberapa kolom (ID, 2 pertanyaan VARK, 2 pernyataan skala 1–5, 2 kolom IPS, IPK). Header ditampilkan memakai kode pendek supaya terbaca; teks lengkapnya ada di tabel 1.3.

In [9]:
contoh_cols = [cols[0], cols[1], cols[2], cols[17], cols[18], cols[108], cols[113], cols[115]]
with pd.option_context("display.max_colwidth", 60):
    display(raw[contoh_cols].rename(columns=nama_baru).head(5))

,ID,VARK_Q01,VARK_Q02,HM_ACT_01,HM_ACT_02,IPS_2023_S2,IPS_2025_S3,IPK
0,1,"melihat diagram, bagan, peta konsep, atau ilustrasi hubu...",meminta seseorang menjelaskan langkah-langkahnya secara ...,3.0,4.0,NaN,NaN,3.78
1,2,"melihat diagram, bagan, peta konsep, atau ilustrasi hubu...",mengikuti gambar langkah demi langkah atau tangkapan lay...,4.0,3.0,NaN,NaN,3.83
2,3,"melihat diagram, bagan, peta konsep, atau ilustrasi hubu...",langsung mencoba sambil memperbaiki kesalahan yang muncu...,5.0,4.0,NaN,NaN,3.77
3,4,"mencoba contoh, simulasi, praktik, atau melihat penerapa...",langsung mencoba sambil memperbaiki kesalahan yang muncul.;,5.0,5.0,NaN,NaN,1.97
4,5,mendengar penjelasan dan dapat berdiskusi atau bertanya ...,mengikuti gambar langkah demi langkah atau tangkapan lay...,4.0,3.0,NaN,NaN,3.54


## 1.5 Data kosong (missing values)
Pertama dilihat per kelompok, lalu per kolom, lalu pola kekosongannya untuk mencari tahu **mengapa** datanya kosong.

In [10]:
def info_kelompok(nama, kolom):
    sub = raw[kolom]
    return {
        "Kelompok": nama,
        "Jumlah kolom": len(kolom),
        "Tipe data": ", ".join(sorted({str(t) for t in sub.dtypes})),
        "Sel kosong": int(sub.isna().sum().sum()),
        "Kolom yg punya sel kosong": int(sub.isna().any().sum()),
        "Nilai unik (min–maks per kolom)": f"{sub.nunique().min()}–{sub.nunique().max()}",
    }

tabel_info = pd.DataFrame([
    info_kelompok("ID", [cols[0]]),
    info_kelompok("VARK (16)", VARK_COLS),
    info_kelompok("Likert 1–5 (88)", LIKERT_COLS),
    info_kelompok("IPS (10)", IPS_COLS),
    info_kelompok("IPK", [IPK_COL]),
])
display(tabel_info)

print("ID unik?", raw[cols[0]].is_unique, "| rentang ID:", raw[cols[0]].min(), "–", raw[cols[0]].max())

,Kelompok,Jumlah kolom,Tipe data,Sel kosong,Kolom yg punya sel kosong,Nilai unik (min–maks per kolom)
0,ID,1,int64,0,0,194–194
1,VARK (16),16,str,32,16,30–41
2,Likert 1–5 (88),88,float64,176,88,4–5
3,IPS (10),10,float64,1118,10,1–139
4,IPK,1,float64,17,1,98–98


ID unik? True | rentang ID: 1 – 194


### a) Kolom survei: dari mana sel kosongnya?

In [11]:
kosong_survei = raw[SURVEY_COLS].isna().sum()
print("Jumlah sel kosong per kolom survei:", sorted(kosong_survei.unique().tolist()),
      f"(berlaku untuk {(kosong_survei > 0).sum()} dari {len(SURVEY_COLS)} kolom)")

baris_kosong_total = raw[SURVEY_COLS].isna().all(axis=1)
print("Baris yang SELURUH jawaban surveinya kosong:", raw.loc[baris_kosong_total, cols[0]].tolist())
print("Baris yang kosong SEBAGIAN:",
      int((raw[SURVEY_COLS].isna().any(axis=1) & ~baris_kosong_total).sum()))
display(raw.loc[baris_kosong_total, [cols[0]] + IPS_COLS + [IPK_COL]].rename(columns=nama_baru))

Jumlah sel kosong per kolom survei: [2] (berlaku untuk 104 dari 104 kolom)
Baris yang SELURUH jawaban surveinya kosong: [11, 133]
Baris yang kosong SEBAGIAN: 0


,ID,IPS_2022_S1,IPS_2022_S2,IPS_2023_S1,IPS_2023_S2,IPS_2024_S1,IPS_2024_S2,IPS_2025_S1,IPS_2025_S2,IPS_2025_S3,IPS_2026_S1,IPK
10,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Temuan:** setiap kolom survei kosong tepat 2 sel, dan semuanya berasal dari **2 baris yang kosong total** (ID 11 dan 133). IPS dan IPK keduanya juga kosong. Tidak ada responden yang mengisi sebagian lalu berhenti. Kedua baris ini dibuang di cleaning (C2).

### b) IPS dan IPK: kosong per kolom

In [12]:
kosong_akad = pd.DataFrame({
    "Terisi": raw[IPS_COLS + [IPK_COL]].notna().sum(),
    "Kosong": raw[IPS_COLS + [IPK_COL]].isna().sum(),
})
kosong_akad["% kosong"] = (kosong_akad["Kosong"] / len(raw) * 100).round(1)
display(kosong_akad.rename(index=nama_baru))

,Terisi,Kosong,% kosong
IPS_2022_S1,1,193,99.5
IPS_2022_S2,1,193,99.5
IPS_2023_S1,18,176,90.7
IPS_2023_S2,18,176,90.7
IPS_2024_S1,125,69,35.6
IPS_2024_S2,125,69,35.6
IPS_2025_S1,173,21,10.8
IPS_2025_S2,172,22,11.3
IPS_2025_S3,16,178,91.8
IPS_2026_S1,173,21,10.8


### c) Pola kekosongan IPS: missing struktural
Kalau kosongnya IPS disebabkan angkatan, polanya pasti teratur: mahasiswa angkatan 2024 punya IPS mulai 2024, angkatan 2025 mulai 2025, dan seterusnya. Tabel di bawah menghitung **tahun IPS pertama** setiap responden (IPS 2026 S1 tidak dihitung karena semester masih berjalan).

In [13]:
ips_selesai_raw = IPS_COLS[:-1]                                   # tanpa IPS 2026 Semester 1
punya_ips = raw[ips_selesai_raw].notna()
tahun_pertama = punya_ips.idxmax(axis=1).str.split().str[1].where(punya_ips.any(axis=1))
pengganti = pd.Series(np.where(raw[IPS_COLS[-1]].notna(), "2026 (hanya IPS semester berjalan)",
                               "tidak ada IPS sama sekali"), index=raw.index)
tahun_pertama = tahun_pertama.fillna(pengganti)
tahun_pertama[baris_kosong_total] = "baris kosong total"

def ada_celah(baris):
    idx = np.where(baris)[0]
    return len(idx) > 0 and (idx[-1] - idx[0] + 1) != len(idx)

pola = (pd.DataFrame({"Tahun IPS pertama": tahun_pertama, "IPK kosong": raw[IPK_COL].isna()})
        .groupby("Tahun IPS pertama").agg(Responden=("IPK kosong", "size"), IPK_kosong=("IPK kosong", "sum")))
display(pola)
print("Responden yang IPS-nya bolong di tengah (cuti?):",
      raw.loc[punya_ips.apply(lambda r: ada_celah(r.to_numpy()), axis=1), cols[0]].tolist())

,Responden,IPK_kosong
Tahun IPS pertama,,
2022,1,0
2023,17,0
2024,107,0
2025,48,0
2026 (hanya IPS semester berjalan),11,7
baris kosong total,2,2
tidak ada IPS sama sekali,8,8


Responden yang IPS-nya bolong di tengah (cuti?): []


**Temuan:** kekosongan IPS mengikuti pola angkatan, jadi termasuk **missing struktural**, bukan responden lupa mengisi. Konsekuensinya:
- IPS **tidak boleh diimputasi** (diisi tebakan), karena nilainya memang belum ada.
- Tahun IPS pertama bisa dipakai untuk **memperkirakan angkatan** (derived variable).
- Responden tanpa IPS sama sekali juga tidak punya IPK, sehingga analisis yang memakai IPK hanya menggunakan responden yang IPK-nya ada.

### d) Kolom yang kosong seluruhnya atau isinya hanya satu nilai
Kolom yang **seluruhnya kosong** tidak berguna sama sekali. Kolom **konstan** (hanya satu nilai) juga tidak memberi informasi, karena semua orang nilainya sama.

In [14]:
kolom_kosong_total = [nama_baru[c] for c in cols if raw[c].isna().all()]
kolom_konstan = pd.DataFrame(
    [(nama_baru[c], raw[c].dropna().unique()[0], int(raw[c].notna().sum())) for c in cols if raw[c].nunique() == 1],
    columns=["Kolom", "Satu-satunya nilai", "Jumlah terisi"],
)
print("Kolom yang seluruhnya kosong:", kolom_kosong_total if kolom_kosong_total else "tidak ada")
print("Kolom yang hanya punya 1 nilai unik:")
display(kolom_konstan)

Kolom yang seluruhnya kosong: tidak ada
Kolom yang hanya punya 1 nilai unik:


,Kolom,Satu-satunya nilai,Jumlah terisi
0,IPS_2022_S1,3.026315,1
1,IPS_2022_S2,3.133333,1
2,IPS_2026_S1,0.000000,173


**Temuan:**
- **Tidak ada kolom yang kosong seluruhnya.**
- **IPS 2026 Semester 1** hanya berisi angka 0 untuk semua yang terisi. Ini bukan IPS nol, tetapi "nilai belum ada" karena semester masih berjalan, sehingga kolom ini tidak informatif.
- **IPS 2022 Semester 1 & 2** hanya terisi oleh 1 orang (satu-satunya angkatan 2022), jadi wajar kalau nilainya hanya satu. Ini bukan masalah format.

## 1.6 Deteksi duplikat
Di tahap ini duplikat hanya **dideteksi**, belum dihapus. Ada empat tingkat pengecekan:
1. **ID ganda:** apakah ada nomor ID yang muncul dua kali?
2. **Duplikat persis:** semua kolom sama (selain ID).
3. **Jawaban survei identik:** ke-104 jawaban survei sama persis.
4. **Sangat mirip:** paling banyak 5 dari 88 jawaban skala 1–5 yang berbeda. Kalau ditemukan, IPS/IPK-nya dibandingkan. Nilai akademik berasal dari data kampus, jadi kalau IPS/IPK berbeda, hampir pasti orangnya berbeda.

In [15]:
id_col = cols[0]
print("1) ID ganda:", int(raw[id_col].duplicated().sum()))

dup_persis = raw.drop(columns=id_col).duplicated(keep=False)
print("2) Baris yang duplikat persis (selain ID):", raw.loc[dup_persis, id_col].tolist())

terisi = raw.loc[~baris_kosong_total]                     # abaikan 2 baris kosong total
dup_survei = terisi[SURVEY_COLS].duplicated(keep=False)
print("3) Jawaban survei identik (di antara baris yang terisi):", terisi.loc[dup_survei, id_col].tolist())

Lk = terisi[LIKERT_COLS].to_numpy(dtype=float)
id_t = terisi[id_col].to_numpy()
mirip = []
for i in range(len(Lk)):
    beda = (Lk[i] != Lk[i + 1:]).sum(axis=1)
    for j in np.where(beda <= 5)[0]:
        a, b = terisi.iloc[i], terisi.iloc[i + 1 + j]
        mirip.append({
            "ID a": id_t[i], "ID b": id_t[i + 1 + j],
            "Likert beda (dari 88)": int(beda[j]),
            "VARK beda (dari 16)": int((a[VARK_COLS] != b[VARK_COLS]).sum()),
            "IPS/IPK sama?": bool(pd.concat([a[IPS_COLS + [IPK_COL]], b[IPS_COLS + [IPK_COL]]], axis=1)
                                  .nunique(axis=1, dropna=False).max() == 1),
        })
tabel_mirip = pd.DataFrame(mirip)
print(f"4) Pasangan sangat mirip: {len(tabel_mirip)} pasang; yang IPS/IPK-nya juga sama: {int(tabel_mirip['IPS/IPK sama?'].sum())}")
display(tabel_mirip.sort_values(["VARK beda (dari 16)", "Likert beda (dari 88)"]).head(10))

1) ID ganda: 0
2) Baris yang duplikat persis (selain ID): [11, 133]
3) Jawaban survei identik (di antara baris yang terisi): []
4) Pasangan sangat mirip: 78 pasang; yang IPS/IPK-nya juga sama: 0


,ID a,ID b,Likert beda (dari 88),VARK beda (dari 16),IPS/IPK sama?
49,78,188,4,1,False
56,125,132,0,3,False
29,55,132,0,10,False
28,55,125,0,11,False
65,136,159,0,11,False
10,38,78,1,11,False
13,38,188,3,11,False
53,115,159,0,12,False
43,68,136,0,14,False
75,167,174,0,14,False


**Mengapa banyak pasangan yang "sangat mirip"?** Ternyata sebagian responden menjawab **angka yang sama untuk seluruh 88 pernyataan** (*straightlining*), atau hampir seluruhnya (84–87 dari 88). Kalau dua orang sama-sama menjawab "4" untuk hampir semua pernyataan, jawaban skala mereka otomatis sangat mirip walaupun orangnya berbeda.

In [16]:
jawaban_terbanyak = terisi[LIKERT_COLS].apply(lambda r: r.value_counts().iloc[0], axis=1)
seragam = terisi[LIKERT_COLS].nunique(axis=1).eq(1)            # 88 dari 88 sama
hampir  = ~seragam & (jawaban_terbanyak >= 84)                  # 84-87 dari 88 sama
ids_seragam = set(terisi.loc[seragam, id_col])
ids_hampir  = set(terisi.loc[hampir, id_col])
print("Straightliner (88 jawaban identik):", len(ids_seragam), "orang ->", sorted(ids_seragam))
print("  angka yang dipilih:", {int(k): int(v) for k, v in terisi.loc[seragam, LIKERT_COLS[0]].value_counts().sort_index().items()})
print("Hampir straightline (84-87 identik):", len(ids_hampir), "orang ->", sorted(ids_hampir))

def jenis_pasangan(r):
    grup = ids_seragam | ids_hampir
    if r["ID a"] in ids_seragam and r["ID b"] in ids_seragam:
        return "keduanya straightliner"
    if r["ID a"] in grup and r["ID b"] in grup:
        return "keduanya (hampir) seragam"
    if r["ID a"] in grup or r["ID b"] in grup:
        return "salah satu (hampir) seragam"
    return "keduanya menjawab bervariasi"

tabel_mirip["Jenis pasangan"] = tabel_mirip.apply(jenis_pasangan, axis=1)
rekap_pasangan = tabel_mirip.groupby("Jenis pasangan").agg(
    Pasangan=("ID a", "size"),
    VARK_beda_min=("VARK beda (dari 16)", "min"),
    IPS_IPK_sama=("IPS/IPK sama?", "sum"),
)
display(rekap_pasangan)

Straightliner (88 jawaban identik): 15 orang -> [9, 38, 55, 57, 64, 68, 115, 125, 132, 136, 159, 167, 171, 174, 194]
  angka yang dipilih: {3: 5, 4: 5, 5: 5}
Hampir straightline (84-87 identik): 8 orang -> [42, 44, 78, 126, 142, 166, 187, 188]


,Pasangan,VARK_beda_min,IPS_IPK_sama
Jenis pasangan,,,
keduanya (hampir) seragam,48,1,0
keduanya straightliner,30,3,0


**Temuan deteksi duplikat:**
- **Tidak ada ID ganda.**
- Duplikat persis hanya muncul pada **ID 11 dan 133**. Keduanya "kembar" karena sama-sama kosong total, bukan karena orang yang sama mengisi dua kali. Keduanya akan dibuang di cleaning sebagai respons kosong.
- **Tidak ada jawaban survei yang identik** di antara responden yang mengisi.
- Semua pasangan yang "sangat mirip" melibatkan responden yang jawabannya **seragam atau hampir seragam**. Tidak ada pasangan mirip di antara responden yang menjawab bervariasi.
- **IPS/IPK semua pasangan berbeda**, dan jawaban VARK-nya pun umumnya berbeda jauh. Pasangan paling mirip, ID 78 & 188 (Likert beda 4, VARK beda 1), tetap punya IPS/IPK berbeda. Kesimpulannya, **tidak ada duplikat responden**.
- Temuan **straightlining** (15 orang) dan **hampir straightline** (8 orang) adalah masalah kualitas jawaban, bukan duplikat. Penanganannya diputuskan di cleaning.

## 1.7 VARK: format jawaban & kategori yang muncul
Setiap jawaban VARK berupa teks opsi yang dipisah tanda `;`, karena responden boleh memilih lebih dari satu. Yang dicek: berapa opsi unik per pertanyaan, berapa opsi yang dipilih orang per pertanyaan, dan opsi apa saja yang muncul.

In [17]:
def pecah(sel):
    """'opsi a;opsi b;' -> ['opsi a', 'opsi b']"""
    return [p.strip() for p in str(sel).split(";") if p.strip()]

rekap_vark = []
for i, c in enumerate(VARK_COLS, 1):
    isi = raw[c].dropna()
    opsi_unik = {o for sel in isi for o in pecah(sel)}
    n_pilih = isi.apply(lambda s: len(pecah(s)))
    rekap_vark.append({
        "Pertanyaan": f"Q{i:02d}",
        "Opsi unik": len(opsi_unik),
        "Min dipilih": n_pilih.min(),
        "Maks dipilih": n_pilih.max(),
        "Rata-rata dipilih": round(n_pilih.mean(), 2),
        "Selalu diakhiri ';'?": bool(isi.str.endswith(";").all()),
    })
display(pd.DataFrame(rekap_vark))

,Pertanyaan,Opsi unik,Min dipilih,Maks dipilih,Rata-rata dipilih,Selalu diakhiri ';'?
0,Q01,4,1,4,2.11,True
1,Q02,4,1,4,1.89,True
2,Q03,4,1,4,1.92,True
3,Q04,4,1,4,1.98,True
4,Q05,4,1,4,1.92,True
5,Q06,4,1,4,2.03,True
6,Q07,4,1,4,2.06,True
7,Q08,4,1,4,1.87,True
8,Q09,4,1,4,1.79,True
9,Q10,4,1,4,1.91,True


**Kategori yang muncul:** semua opsi yang ada di setiap pertanyaan, beserta berapa responden yang memilihnya. Karena boleh memilih lebih dari satu, total persentase per pertanyaan bisa lebih dari 100%.

In [18]:
n_jawab = raw[VARK_COLS[0]].notna().sum()
frek = []
for i, c in enumerate(VARK_COLS, 1):
    hitung = pd.Series([o for s in raw[c].dropna() for o in pecah(s)]).value_counts()
    for opsi, n in hitung.items():
        frek.append({"Pertanyaan": f"Q{i:02d}", "Opsi": opsi.rstrip("."), "Dipilih": int(n),
                     "% responden": round(n / n_jawab * 100, 1)})
frek_vark = pd.DataFrame(frek)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(frek_vark)

,Pertanyaan,Opsi,Dipilih,% responden
0,Q01,"mencoba contoh, simulasi, praktik, atau melihat penerapan nyata",121,63.0
1,Q01,mendengar penjelasan dan dapat berdiskusi atau bertanya secara lisan,113,58.9
2,Q01,"melihat diagram, bagan, peta konsep, atau ilustrasi hubungan antarbagian",92,47.9
3,Q01,"membaca uraian tertulis, definisi, atau catatan yang terstruktur",80,41.7
4,Q02,mengikuti gambar langkah demi langkah atau tangkapan layar,107,55.7
5,Q02,membaca petunjuk tertulis dan daftar langkah,94,49.0
6,Q02,langsung mencoba sambil memperbaiki kesalahan yang muncul,89,46.4
7,Q02,meminta seseorang menjelaskan langkah-langkahnya secara lisan,72,37.5
8,Q03,"berlatih soal, mencoba contoh, atau menghubungkan materi dengan pengalaman",131,68.2
9,Q03,"menulis ulang ringkasan, membuat daftar, atau membaca catatan",128,66.7


**Temuan:** setiap pertanyaan punya **tepat 4 opsi**, masing-masing mewakili satu modalitas V/A/R/K (pemetaannya ditetapkan di C4). Tidak ada isian bebas atau typo, dan orang memilih 1 sampai 4 opsi per pertanyaan. Opsi yang paling sering dipilih di antaranya "menjelaskan secara lisan" (Q12) dan "berlatih soal/mencoba contoh" (Q03).

## 1.8 Pernyataan skala 1–5: rentang, distribusi, dan statistik ringkas

In [19]:
nilai = pd.Series(raw[LIKERT_COLS].values.ravel()).dropna()
print("Nilai minimum:", nilai.min(), "| maksimum:", nilai.max())
print("Ada nilai desimal?", bool((nilai % 1 != 0).any()))

dist = nilai.value_counts().sort_index().rename("jumlah").to_frame()
dist["persen"] = (dist["jumlah"] / dist["jumlah"].sum() * 100).round(1)
display(dist)

Nilai minimum: 1.0 | maksimum: 5.0
Ada nilai desimal? False


,jumlah,persen
1.0,166,1.0
2.0,710,4.2
3.0,4387,26.0
4.0,6862,40.6
5.0,4771,28.2


**Statistik ringkas per pernyataan** (88 item). Kolom `% setuju` = persentase jawaban 4 atau 5.

In [20]:
L = raw[LIKERT_COLS]
stat_likert = pd.DataFrame({
    "Kode": [nama_baru[c] for c in LIKERT_COLS],
    "Pernyataan": [c[:75] for c in LIKERT_COLS],
    "n": L.notna().sum().values,
    "Rata-rata": L.mean().round(2).values,
    "SD": L.std().round(2).values,
    "Median": L.median().values,
    "Min": L.min().values,
    "Maks": L.max().values,
    "Nilai unik": L.nunique().values,
    "% setuju": ((L >= 4).sum() / L.notna().sum() * 100).round(1).values,
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    display(stat_likert)

print("Rata-rata item terendah :", stat_likert.nsmallest(1, "Rata-rata")[["Kode", "Rata-rata"]].values.tolist())
print("Rata-rata item tertinggi:", stat_likert.nlargest(1, "Rata-rata")[["Kode", "Rata-rata"]].values.tolist())
print("Rentang % setuju per item:", stat_likert["% setuju"].min(), "–", stat_likert["% setuju"].max())

,Kode,Pernyataan,n,Rata-rata,SD,Median,Min,Maks,Nilai unik,% setuju
0,HM_ACT_01,Saya menikmati aktivitas belajar baru meskipun belum sepenuhnya tahu bagaim,192,3.79,0.88,4.0,1.0,5.0,5,66.1
1,HM_ACT_02,Saya lebih bersemangat ketika dapat langsung terlibat dalam tugas atau tant,192,3.89,0.88,4.0,1.0,5.0,5,65.1
2,HM_ACT_03,Saya suka mencoba berbagai cara tanpa terlalu lama menunggu sebelum memulai,192,3.71,0.93,4.0,1.0,5.0,5,57.3
3,HM_ACT_04,"Diskusi, simulasi, atau aktivitas kelompok membuat saya lebih aktif belajar",192,3.85,0.87,4.0,1.0,5.0,5,67.7
4,HM_ACT_05,Saya tertarik pada pengalaman yang menuntut spontanitas dan keterlibatan la,192,3.94,0.89,4.0,1.0,5.0,5,70.3
5,HM_ACT_06,Saya cenderung menerima tantangan baru sebagai kesempatan untuk belajar.,192,3.83,0.91,4.0,1.0,5.0,5,64.1
6,HM_ACT_07,Saya merasa belajar paling hidup ketika banyak hal terjadi dan saya dapat i,192,3.98,0.88,4.0,1.0,5.0,5,70.3
7,HM_ACT_08,Saya mudah tertarik mencoba metode atau alat baru dalam pembelajaran.,192,3.93,0.84,4.0,1.0,5.0,5,71.4
8,HM_REF_01,Saya suka mengamati terlebih dahulu sebelum mengambil kesimpulan.,192,4.18,0.81,4.0,2.0,5.0,4,79.7
9,HM_REF_02,Saya mempertimbangkan pengalaman dari beberapa sudut pandang sebelum bertin,192,4.09,0.83,4.0,1.0,5.0,5,76.6


Rata-rata item terendah : [['HM_REF_06', 3.26]]
Rata-rata item tertinggi: [['HM_THE_03', 4.24]]
Rentang % setuju per item: 38.5 – 82.3


**Temuan:** semua jawaban bulat 1–5, tanpa nilai aneh. Sekitar **69% jawaban adalah 4 atau 5**, dan rata-rata setiap item berada di kisaran 3,3–4,2. Responden cenderung menjawab "setuju", dan ini penting saat menafsirkan skor nanti. Beberapa item tidak pernah dijawab 1 sehingga hanya punya 4 nilai unik. Itu wajar dan bukan error.

## 1.9 IPS dan IPK

In [21]:
display(raw[IPS_COLS + [IPK_COL]].describe().T.round(2))
print("Nilai IPS 2026 Semester 1 yang muncul:", raw[IPS_COLS[-1]].value_counts(dropna=False).to_dict())

,count,mean,std,min,25%,50%,75%,max
IPS 2022 Semester 1,1.0,3.03,NaN,3.03,3.03,3.03,3.03,3.03
IPS 2022 Semester 2,1.0,3.13,NaN,3.13,3.13,3.13,3.13,3.13
IPS 2023 Semester 1,18.0,3.49,0.38,2.68,3.20,3.62,3.73,3.97
IPS 2023 Semester 2,18.0,3.55,0.33,2.83,3.36,3.56,3.80,4.00
IPS 2024 Semester 1,125.0,3.33,0.37,2.26,3.13,3.34,3.59,4.00
IPS 2024 Semester 2,125.0,3.37,0.50,0.95,3.18,3.44,3.71,4.00
IPS 2025 Semester 1,173.0,3.29,0.56,0.87,3.01,3.40,3.72,4.00
IPS 2025 Semester 2,172.0,3.47,0.46,0.94,3.22,3.59,3.82,4.00
IPS 2025 Semester 3,16.0,3.47,0.43,3.00,3.00,3.50,3.89,4.00
IPS 2026 Semester 1,173.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Nilai IPS 2026 Semester 1 yang muncul: {0.0: 173, nan: 21}


**Temuan:**
- Semua nilai berada di rentang 0–4.
- **IPS 2026 Semester 1 bernilai 0 untuk semua orang** karena semester ini masih berjalan. Angka 0 di sini berarti "belum ada nilai", bukan IPS nol.
- Ada IPK = 0 pada mahasiswa baru yang belum punya IPS sama sekali.

Keduanya akan diperbaiki di tahap cleaning.

## 1.10 Ringkasan: variabel dikelompokkan berdasarkan peran
Spesifikasi (Bagian 7.1) meminta variabel dikelompokkan berdasarkan **peran/jenisnya**: metadata non-identifying, item survei, skor numerik, dan label/kategori. Tabel di bawah juga merangkum nilai unik, kategori yang muncul, rentang, dan konsistensi format setiap kelompok.

In [22]:
def rentang(kolom):
    v = raw[kolom].stack()
    return f"{v.min():g} – {v.max():g}"

def desimal_maks(kolom):
    v = raw[kolom].stack()
    return int(v.map(lambda x: len(f"{x}".split(".")[1].rstrip("0")) if "." in f"{x}" else 0).max())

opsi_per_soal = sorted({len({o for s in raw[c].dropna() for o in pecah(s)}) for c in VARK_COLS})
kombinasi = raw[VARK_COLS].nunique()

peran = pd.DataFrame([
    {
        "Peran": "Identifier (bukan PII)",
        "Kolom": "ID", "Jumlah": 1, "Tipe": "integer",
        "Nilai unik / kategori": f"{raw[cols[0]].nunique()} nilai unik (1 per baris)",
        "Rentang": rentang([cols[0]]),
        "Konsistensi format": "Unik, berurutan, tanpa celah; hanya nomor urut",
    },
    {
        "Peran": "Metadata non-identifying (konteks akademik)",
        "Kolom": "IPS per semester, IPK", "Jumlah": len(IPS_COLS) + 1, "Tipe": "numerik kontinu",
        "Nilai unik / kategori": f"IPS {raw[IPS_COLS].nunique().min()}–{raw[IPS_COLS].nunique().max()} per kolom; IPK {raw[IPK_COL].nunique()}",
        "Rentang": f"IPS {rentang(IPS_COLS)}; IPK {rentang([IPK_COL])}",
        "Konsistensi format": f"IPS sampai {desimal_maks(IPS_COLS)} desimal, IPK {desimal_maks([IPK_COL])} desimal; "
                              "IPS 2026 S1 semua 0 (semester berjalan); IPK 0 pada mhs baru",
    },
    {
        "Peran": "Item survei – kategorikal multi-jawab",
        "Kolom": "VARK_Q01–Q16", "Jumlah": len(VARK_COLS), "Tipe": "teks (opsi dipisah ';')",
        "Nilai unik / kategori": f"{opsi_per_soal[0]} opsi per soal (V/A/R/K); {kombinasi.min()}–{kombinasi.max()} kombinasi jawaban per kolom",
        "Rentang": "1–4 opsi dipilih per soal",
        "Konsistensi format": "Semua diakhiri ';', tanpa typo/isian bebas",
    },
    {
        "Peran": "Item survei – ordinal (Likert 1–5)",
        "Kolom": "HM/FS/KOLB (88 pernyataan)", "Jumlah": len(LIKERT_COLS), "Tipe": "ordinal (tersimpan float)",
        "Nilai unik / kategori": f"{raw[LIKERT_COLS].nunique().min()}–{raw[LIKERT_COLS].nunique().max()} nilai per kolom dari {{1,2,3,4,5}}",
        "Rentang": rentang(LIKERT_COLS),
        "Konsistensi format": "Semua bulat, tanpa nilai di luar 1–5",
    },
    {
        "Peran": "Skor numerik",
        "Kolom": "—", "Jumlah": 0, "Tipe": "—",
        "Nilai unik / kategori": "Belum ada di dataset asli",
        "Rentang": "—",
        "Konsistensi format": "Dibuat di Bagian 3 (scoring sesuai teori)",
    },
    {
        "Peran": "Label / kategori gaya belajar",
        "Kolom": "—", "Jumlah": 0, "Tipe": "—",
        "Nilai unik / kategori": "Belum ada di dataset asli",
        "Rentang": "—",
        "Konsistensi format": "Dibuat di Bagian 3 (scoring sesuai teori)",
    },
])
assert peran["Jumlah"].sum() == raw.shape[1]
display(peran)

,Peran,Kolom,Jumlah,Tipe,Nilai unik / kategori,Rentang,Konsistensi format
0,Identifier (bukan PII),ID,1,integer,194 nilai unik (1 per baris),1 – 194,"Unik, berurutan, tanpa celah; hanya nomor urut"
1,Metadata non-identifying (konteks akademik),"IPS per semester, IPK",11,numerik kontinu,IPS 1–139 per kolom; IPK 98,IPS 0 – 4; IPK 0 – 4,"IPS sampai 6 desimal, IPK 2 desimal; IPS 2026 S1 semua 0 (semester berjalan)..."
2,Item survei – kategorikal multi-jawab,VARK_Q01–Q16,16,teks (opsi dipisah ';'),4 opsi per soal (V/A/R/K); 30–41 kombinasi jawaban per kolom,1–4 opsi dipilih per soal,"Semua diakhiri ';', tanpa typo/isian bebas"
3,Item survei – ordinal (Likert 1–5),HM/FS/KOLB (88 pernyataan),88,ordinal (tersimpan float),"4–5 nilai per kolom dari {1,2,3,4,5}",1 – 5,"Semua bulat, tanpa nilai di luar 1–5"
4,Skor numerik,—,0,—,Belum ada di dataset asli,—,Dibuat di Bagian 3 (scoring sesuai teori)
5,Label / kategori gaya belajar,—,0,—,Belum ada di dataset asli,—,Dibuat di Bagian 3 (scoring sesuai teori)


**Kesimpulan data understanding:**
- Dataset berisi **item survei mentah** (16 VARK + 88 Likert) ditambah **metadata akademik** (IPS, IPK) dan **ID**. Totalnya 116 kolom.
- **Belum ada skor numerik maupun label.** Keduanya harus dibuat sendiri dengan aturan teori masing-masing framework (Bagian 3).
- **Tidak ada metadata demografis** seperti prodi, angkatan, atau waktu pengisian. Angkatan hanya bisa diperkirakan dari tahun IPS pertama, dan perkiraan ini akan diperlakukan sebagai *derived variable*.
- Formatnya konsisten. Masalah yang perlu dibersihkan hanya 2 baris kosong total dan angka 0 yang sebenarnya berarti "belum ada nilai" di IPS/IPK.

## 1.11 Ringkasan temuan & apa yang akan dilakukan di cleaning
Bagian ini hanya **mendeteksi**. Semua perbaikan dikerjakan di Bagian 2 (cleaning) dan dicatat di cleaning log.

| # | Temuan data understanding | Bagian | Rencana di cleaning |
|---|---|---|---|
| 1 | Header berupa teks pertanyaan yang panjang | 1.3 | Ganti nama ke kode pendek; teks asli disimpan untuk data dictionary |
| 2 | 2 baris kosong total (ID 11, 133), yang juga terdeteksi sebagai "duplikat" satu sama lain | 1.5, 1.6 | **Hapus** (respons kosong) |
| 3 | Pernyataan skala 1–5 tersimpan sebagai desimal | 1.3, 1.8 | Validasi nilai 1–5, ubah tipe ke bilangan bulat |
| 4 | Jawaban VARK diakhiri `;`, 4 opsi per soal | 1.7 | Validasi opsi terhadap daftar resmi, rapikan format |
| 5 | IPS 2026 S1 = 0 untuk semua (semester berjalan) | 1.5d, 1.9 | Ubah 0 → kosong (`NaN`) |
| 6 | IPK = 0 pada mahasiswa baru yang belum punya IPS | 1.9 | Ubah 0 → kosong agar konsisten |
| 7 | IPS banyak kosong karena angkatan (*missing struktural*) | 1.5c | **Tidak diimputasi**; tahun IPS pertama dipakai memperkirakan angkatan |
| 8 | Beberapa IPS/IPK sangat rendah | 1.9 | **Dipertahankan** (nilai valid yang ekstrem) |
| 9 | Tidak ada duplikat responden | 1.6 | Tidak ada aksi; hasil pengecekan dicatat |
| 10 | 15 responden menjawab 88 pernyataan dengan angka yang sama, 8 lainnya hampir sama | 1.6 | **Diberi tanda (flag)**, tidak dihapus |
| 11 | Belum ada skor maupun label gaya belajar | 1.10 | Dibuat di Bagian 3 (scoring sesuai teori) |

---
# Bagian 2 – Data Cleaning

Bagian ini menangani masalah yang sudah dideteksi di Bagian 1 (lihat tabel 1.11).
Setiap langkah cleaning:
1. dikerjakan lewat kode,
2. dicatat ke `cleaning_log`: langkah, alasan, jumlah baris sebelum/sesudah, dan data yang terdampak.

Data asli (`raw`) tidak diubah. Semua perubahan dilakukan pada salinannya (`df`).

In [23]:
cleaning_log = []

def catat(langkah, aksi, alasan, n_sebelum, n_sesudah, terdampak=""):
    """Mencatat satu langkah cleaning ke dalam cleaning_log."""
    cleaning_log.append({
        "Langkah": langkah,
        "Aksi": aksi,
        "Alasan": alasan,
        "Baris sebelum": n_sebelum,
        "Baris sesudah": n_sesudah,
        "Terdampak": terdampak,
    })

df = raw.copy()   # semua cleaning dikerjakan pada salinan; `raw` tetap asli

print("Salinan data dibuat:", df.shape[0], "baris x", df.shape[1], "kolom")

Salinan data dibuat: 194 baris x 116 kolom


## C1 – Ganti nama kolom
**Masalah (temuan 1.3):** header kolom berupa teks pertanyaan yang panjang, sehingga sulit dipakai di kode.

**Penanganan:** ganti ke kode pendek yang sudah disusun di Bagian 1.2. Teks aslinya disimpan di tabel `kolom_map` supaya setiap kolom tetap bisa dilacak ke pertanyaan aslinya (Spesifikasi 7.3).

| Pola nama | Contoh | Arti |
|---|---|---|
| `VARK_Qxx` | `VARK_Q01` | Pertanyaan VARK ke-1 |
| `<FRAMEWORK>_<TIPE>_xx` | `FS_SEN_03` | Felder-Silverman, Sensing, pernyataan ke-3 |
| `IPS_<tahun>_S<n>` | `IPS_2025_S2` | IPS tahun 2025 semester 2 |

In [24]:
# 1) Simpan pasangan nama baru <-> teks asli (dipakai nanti di data dictionary)
kolom_map = pd.DataFrame({
    "nama_kolom": [nama_baru[c] for c in cols],
    "teks_asli": cols,
    "framework": [fw for nama, fw, a, b, kode in GROUPS for _ in range(a, b + 1)],
})

# 2) Ganti nama kolom di salinan data
df = df.rename(columns=nama_baru)
assert df.columns.is_unique, "Ada nama kolom baru yang kembar!"

# 3) Perbarui daftar kolom per kelompok supaya memakai nama baru
VARK_COLS   = [c for c in df.columns if c.startswith("VARK_Q")]
LIKERT_COLS = [c for c in df.columns if c.startswith(("HM_", "FS_", "KOLB_"))]
IPS_COLS    = [c for c in df.columns if c.startswith("IPS_")]
SURVEY_COLS = VARK_COLS + LIKERT_COLS

# 4) Catat ke cleaning log
catat("C1", "Ganti nama 116 kolom ke kode pendek",
      "Header asli terlalu panjang; teks asli disimpan di kolom_map",
      len(df), len(df))

print("Jumlah kolom:", len(df.columns))
print(f"VARK: {len(VARK_COLS)} | Likert: {len(LIKERT_COLS)} | IPS: {len(IPS_COLS)}")
print("Kolom IPS:", IPS_COLS)
display(kolom_map.iloc[[0, 1, 17, 49, 81, 105, 115]])

Jumlah kolom: 116
VARK: 16 | Likert: 88 | IPS: 10
Kolom IPS: ['IPS_2022_S1', 'IPS_2022_S2', 'IPS_2023_S1', 'IPS_2023_S2', 'IPS_2024_S1', 'IPS_2024_S2', 'IPS_2025_S1', 'IPS_2025_S2', 'IPS_2025_S3', 'IPS_2026_S1']


,nama_kolom,teks_asli,framework
0,ID,ID,Identifier
1,VARK_Q01,"Ketika mempelajari topik baru yang cukup kompleks, saya paling terbantu jika...",VARK
17,HM_ACT_01,Saya menikmati aktivitas belajar baru meskipun belum sepenuhnya tahu bagaima...,Honey-Mumford
49,FS_ACT_01,Saya memahami materi lebih cepat ketika dapat mendiskusikannya atau mengerja...,Felder-Silverman
81,KOLB_CE_01,Saya lebih mudah mulai belajar ketika memiliki pengalaman atau contoh konkre...,Kolb
105,IPS_2022_S1,IPS 2022 Semester 1,Nilai akademik
115,IPK,IPK,Nilai akademik


## C2 – Buang respons kosong total
**Masalah (temuan 1.5a dan 1.6):** ID 11 dan 133 tidak berisi jawaban survei sama sekali, dan IPS/IPK-nya juga kosong. Keduanya juga terdeteksi sebagai "duplikat persis" satu sama lain, padahal itu hanya karena sama-sama kosong.

**Penanganan:** kedua baris dibuang. Tidak ada data yang bisa dianalisis dari baris ini, dan Spesifikasi 7.2 menyebut "respons kosong" sebagai hal yang perlu ditangani. Jumlah sebelum/sesudah dan ID-nya dicatat di cleaning log.

In [25]:
n_sebelum = len(df)

# 1) Cari baris yang SELURUH 104 jawaban surveinya kosong
kosong_total = df[SURVEY_COLS].isna().all(axis=1)
id_kosong = df.loc[kosong_total, "ID"].tolist()

# 2) Tampilkan dulu baris yang akan dibuang (sebagai bukti)
print("Baris yang akan dibuang:")
display(df.loc[kosong_total, ["ID"] + IPS_COLS + ["IPK"]])

# 3) Buang, lalu rapikan nomor index
df = df.loc[~kosong_total].reset_index(drop=True)

# 4) Catat ke cleaning log
catat("C2", "Hapus baris yang seluruh jawaban surveinya kosong",
      "Respons kosong total (survei, IPS, dan IPK kosong); tidak ada data untuk dianalisis",
      n_sebelum, len(df), f"ID {id_kosong}")

print(f"\nJumlah baris: {n_sebelum} -> {len(df)} (dibuang: ID {id_kosong})")
print("Sisa sel kosong di kolom survei:", int(df[SURVEY_COLS].isna().sum().sum()))

Baris yang akan dibuang:


,ID,IPS_2022_S1,IPS_2022_S2,IPS_2023_S1,IPS_2023_S2,IPS_2024_S1,IPS_2024_S2,IPS_2025_S1,IPS_2025_S2,IPS_2025_S3,IPS_2026_S1,IPK
10,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Jumlah baris: 194 -> 192 (dibuang: ID [11, 133])
Sisa sel kosong di kolom survei: 0


## C3 – Validasi jawaban skala 1–5 & perbaiki tipe data
**Masalah (temuan 1.3 dan 1.8):** 88 pernyataan skala 1–5 tersimpan sebagai angka desimal (`float64`, misalnya `4.0`), padahal jawabannya bilangan bulat. Penyebabnya, pandas otomatis memakai desimal untuk kolom angka yang punya sel kosong.

**Penanganan:**
1. Validasi dulu bahwa semua jawaban adalah 1, 2, 3, 4, atau 5.
2. Ubah tipe data menjadi bilangan bulat (`Int64`).

Ini **hanya konversi format penyimpanan**. Nilai jawaban tidak berubah (`4.0` → `4`), sehingga makna jawaban asli tetap terjaga (Spesifikasi 7.3).

In [26]:
# 1) Validasi: semua jawaban harus salah satu dari 1, 2, 3, 4, 5
tidak_valid = ~df[LIKERT_COLS].isin([1, 2, 3, 4, 5])
n_tidak_valid = int(tidak_valid.sum().sum())
print("Jumlah jawaban di luar 1-5:", n_tidak_valid)
assert n_tidak_valid == 0, "Ada jawaban di luar 1-5, periksa dulu sebelum lanjut!"

# 2) Simpan salinan nilai sebelum konversi (untuk bukti bahwa nilainya tidak berubah)
sebelum = df[LIKERT_COLS].copy()

# 3) Ubah tipe data: desimal (float64) -> bilangan bulat (Int64)
print("Tipe data sebelum:", df[LIKERT_COLS].dtypes.unique())
df[LIKERT_COLS] = df[LIKERT_COLS].astype("Int64")
print("Tipe data sesudah:", df[LIKERT_COLS].dtypes.unique())

# 4) Bukti: nilai jawaban sama persis sebelum dan sesudah konversi
print("Nilai jawaban berubah?", not (sebelum == df[LIKERT_COLS].astype(float)).all().all())
display(pd.concat([sebelum.iloc[:3, :4].add_suffix(" (sebelum)"),
                   df[LIKERT_COLS].iloc[:3, :4].add_suffix(" (sesudah)")], axis=1))

# 5) Catat ke cleaning log
catat("C3", "Validasi 88 item Likert & ubah tipe float64 -> Int64",
      f"{n_tidak_valid} nilai di luar 1-5; konversi format saja, nilai jawaban tidak berubah",
      len(df), len(df))

Jumlah jawaban di luar 1-5: 0
Tipe data sebelum: [dtype('float64')]
Tipe data sesudah: [Int64Dtype()]
Nilai jawaban berubah? False


,HM_ACT_01 (sebelum),HM_ACT_02 (sebelum),HM_ACT_03 (sebelum),HM_ACT_04 (sebelum),HM_ACT_01 (sesudah),HM_ACT_02 (sesudah),HM_ACT_03 (sesudah),HM_ACT_04 (sesudah)
0,3.0,4.0,1.0,3.0,3,4,1,3
1,4.0,3.0,3.0,4.0,4,3,3,4
2,5.0,4.0,4.0,3.0,5,4,4,3
